In [1]:
import pandas as pd
import os
import numpy as np
import re
from datetime import datetime, timedelta

In [2]:
pd.set_option('display.max_columns', None)

# **Step 1. Merge MO cancer patients with EHR**
### output: <span style="color:DodgerBlue;">**mo_cancer_candidate_ehr** & **mo_cancer_candidate_ehr_@index_1**</span>

#### <span style="color:#FF6347;">**READ**</span> mo_cancer_candidate file

In [3]:
# file_path = "P:/Dataset/MO-DBT-data-curation/Cancer/mo_cancer.xlsx"
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "mo_cancer_candidate" + ".xlsx")
df = pd.read_excel(file_path)

In [4]:
df.head(3)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,EXAM_NAME,Study,Series,needs_biopsy,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE,Index,time_to_INDEX_days,time_to_INDEX_months,path_to_INDEX_months
0,4330344296,62861703,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-12-20,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,False,NaN,NaN,NaN,L,INDEX-1,377.0,12.4,NaN
1,4330344296,62861703,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-12-20,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,False,NaN,NaN,NaN,R,INDEX-1,377.0,12.4,NaN
2,4330344296,67469599,Heterogeneously dense (51% - 75%),0 - Need additional imaging evaluation,P-Additional projections,2020-12-31,1,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,False,NaN,NaN,NaN,NaN,INDEX,NaN,NaN,NaN


In [5]:
df.columns

Index(['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'COMPOSITION_NAME',
       'FINDING_CATEGORY', 'FINDING_REC', 'EXAM_COMPLETED_DATE',
       'duplicate_count', 'EXAM_NAME', 'Study', 'Series', 'needs_biopsy',
       'BX_ID', 'PATHOLOGY_DATE', 'LESION_CLASS', 'SIDE', 'Index',
       'time_to_INDEX_days', 'time_to_INDEX_months', 'path_to_INDEX_months'],
      dtype='object')

In [6]:
extract_cols = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE', 'Study', 'Index']

In [7]:
df_patient_visit = df.drop_duplicates(
    subset=extract_cols, keep = 'first').reset_index(drop = True)[extract_cols]

In [8]:
df_patient_visit.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,EXAM_COMPLETED_DATE,Study,Index
0,4330344296,62861703,2019-12-20,SCREEN,INDEX-1
1,4330344296,67469599,2020-12-31,SCREEN,INDEX
2,4330344296,67156697,2021-01-25,DIAG,INDEX
3,4330358672,78682610,2018-05-21,SCREEN,INDEX-1
4,4330358672,63576941,2019-06-14,DIAG,INDEX


#### <span style="color:#FF6347;">**READ**</span> **EHR** (hormonal_mens, risk_factors, patient_demo, vitals )

In [9]:
hormonal_mens = pd.read_excel(os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "Cleaned", "hormonal_mens" + ".xlsx"))
risk_factors = pd.read_excel(os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "Cleaned", "risk_factors" + ".xlsx"))
patient_demo = pd.read_excel(os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "Cleaned", "patient_demo" + ".xlsx"))
vitals = pd.read_excel(os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "Cleaned", "vitals" + ".xlsx"))

### **1. Merge BMI from vitals**

In [10]:
vitals.head(5)

,PATIENT_STUDY_ID,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI
0,4330018595,2019-06-24,2208.0,OZ,63.5,IN,24.06
1,4330018595,2020-03-03,2176.0,OZ,63.5,IN,23.71
2,4330018595,2020-06-23,2000.0,OZ,63.5,IN,21.80
3,4330018595,2020-07-07,2000.0,OZ,63.5,IN,21.80
4,4330018595,2020-08-11,2000.0,OZ,63.5,IN,21.80


In [11]:
df_patient_visit['EXAM_COMPLETED_DATE'] = pd.to_datetime(df_patient_visit['EXAM_COMPLETED_DATE'])
vitals['DATE_TAKEN']                    = pd.to_datetime(vitals['DATE_TAKEN'])

visit_sorted  = df_patient_visit.sort_values('EXAM_COMPLETED_DATE').reset_index(drop=True)
vitals_sorted = vitals.sort_values('DATE_TAKEN').reset_index(drop=True)

df_patient_visit_vital = pd.merge_asof(
    visit_sorted,
    vitals_sorted,
    left_on  = 'EXAM_COMPLETED_DATE',
    right_on = 'DATE_TAKEN',
    by       = 'PATIENT_STUDY_ID',
    # tolerance=pd.Timedelta(days=365),
    direction= 'nearest'
)

print(f"df_patient_visit : {len(df_patient_visit)} rows")
print(f"df_patient_visit_vital : {len(df_patient_visit_vital)} rows")

df_patient_visit : 1966 rows
df_patient_visit_vital : 1966 rows


In [12]:
df_patient_visit_vital.sort_values(['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], ignore_index=True, inplace=True)

In [13]:
df_patient_visit_vital.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,EXAM_COMPLETED_DATE,Study,Index,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI
0,4330344296,62861703,2019-12-20,SCREEN,INDEX-1,2019-09-06,2320.0,OZ,60.0,IN,28.32
1,4330344296,67156697,2021-01-25,DIAG,INDEX,2021-02-23,2528.0,OZ,60.0,IN,30.86
2,4330344296,67469599,2020-12-31,SCREEN,INDEX,2021-02-23,2528.0,OZ,60.0,IN,30.86
3,4330358672,63576941,2019-06-14,DIAG,INDEX,2019-06-24,2352.0,OZ,68.0,IN,22.35
4,4330358672,78682610,2018-05-21,SCREEN,INDEX-1,2018-07-09,2320.0,OZ,68.0,IN,22.05


### **2. Merge hormonal_mens, risk_factors, patient_demo**

In [14]:
merge_hr = hormonal_mens.merge(risk_factors, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE'], how='inner')
merge_hrp = merge_hr.merge(patient_demo, on=['PATIENT_STUDY_ID'], how='inner')

### **2.1. Calculate Age at exam date**

#### <span style="color:darkcyan">**Function**</span>

In [15]:
from pandas import Int64Dtype
def calculate_age_at_study(df):
    """
    Calculates the patient's age in years at the time of the study 
    based on the 'PatientBirthDate' and 'StudyDate' columns.
    """
    
    # 1. Ensure date columns are in datetime format
    # The format 'YYYY-MM-DD' is used for parsing
    try:
        df['BIRTH_DATE_DT'] = pd.to_datetime(df['BIRTH_DATE'], format='%Y-%m-%d')
        # Note: The StudyDate column often includes time (e.g., '2020-06-01 09:10:52').
        # We can let pandas infer the format for this one since it's cleaner.
        df['EXAM_COMPLETED_DATE_DT'] = pd.to_datetime(df['EXAM_COMPLETED_DATE'], errors='coerce') 
    except ValueError as e:
        print(f"Error parsing date format: {e}. Please check your date column formats.")
        return df

    # 2. Calculate the difference in days
    time_difference = df['EXAM_COMPLETED_DATE_DT'] - df['BIRTH_DATE_DT']
    
    # 3. Convert the difference into whole years (integer format)
    # The .dt.days attribute gives the number of days, which is divided by 365.25 
    # and then explicitly cast to an integer to capture only the full years elapsed.
    mask = df['BIRTH_DATE_DT'].notna() & df['EXAM_COMPLETED_DATE_DT'].notna()
    df.loc[mask, 'PatientAge'] = (time_difference[mask].dt.days / 365.25).astype(int)
    # df['PatientAge'] = (time_difference.dt.days / 365.25).astype('Int64')
    
    # Clean up the intermediate columns
    
    df = df.drop(columns=['BIRTH_DATE_DT', 'EXAM_COMPLETED_DATE_DT'])
    
    return df

#### <span style="color:blue">**Main**</span>

In [16]:
merge_hrp = calculate_age_at_study(merge_hrp)

In [17]:
merge_hrp.head(3)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,EXAM_COMPLETED_DATE,RISK_FACTOR_No family history of breast cancer,RISK_FACTOR_Personal breast cancer history,"RISK_FACTOR_Weak family history of breast cancer (aunt, grandmother, cousin)",RISK_FACTOR_Late child bearing (after 30),"RISK_FACTOR_Very strong family history of breast cancer (mother, sister, daughter, pre-menopause or multiple post-menopausal first degree relatives)","RISK_FACTOR_Family history of ovarian cancer in mother, sister, or daughter",RISK_FACTOR_Post-menopausal patient,RISK_FACTOR_BRCA1 gene mutation,RISK_FACTOR_Nulliparous,RISK_FACTOR_Family history of ovarian cancer in distant blood relatives,"RISK_FACTOR_Intermediate family history of breast cancer (mother, sister, daughter, post-menopause)","RISK_FACTOR_History of high risk lesion on previous biopsy, LCIS, atypical hyperplasia",RISK_FACTOR_Previous chest radiation therapy,RISK_FACTOR_Other family history of breast cancer,RISK_FACTOR_BRCA2 gene mutation,RISK_FACTOR_Family history unknown for breast cancer,RISK_FACTOR_History of ovarian cancer,RISK_FACTOR_History of endometrial cancer,RISK_FACTOR_Not available,BIRTH_DATE,GENDER_TITLE,RACE_KOREAN,RACE_WHITE,RACE_INDIAN (ASIAN),RACE_BLACK,RACE_UNREPORTED/CHOSE NOT TO DISCLOSE,RACE_OTHER ASIAN,RACE_VIETNAMESE,RACE_CHINESE,RACE_AMERICAN INDIAN/ALASKA NATIVE,RACE_FILIPINO,RACE_OTHER,RACE_JAPANESE,RACE_HAWAIIAN,RACE_OTHER PACIFIC ISLANDER,ETHNIC_NOT HISPANIC OR LATINO,"ETHNIC_ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN",ETHNIC_UNREPORTED/CHOSE NOT TO DISCLOSE,ETHNIC_HISPANIC OR LATINO,"ETHNIC_MEXICAN, MEXICAN AMERICAN, CHICANO",ETHNIC_PUERTO RICAN,PatientAge
0,4330018595,63027507,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2019-07-26,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1965-07-01,FEMALE,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,54.0
1,4330018595,63737104,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2019-08-19,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1965-07-01,FEMALE,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,54.0
2,4330018595,60103700,16,31,49,0,0,0,2.0,3.0,NaN,NaN,2020-06-02,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1965-07-01,FEMALE,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,54.0


### **3. Merge MO cancer patients with EHR**

In [18]:
merge_hrp['EXAM_COMPLETED_DATE'] = pd.to_datetime(merge_hrp['EXAM_COMPLETED_DATE'])

In [19]:
df_patient_visit_ehr = pd.merge(df_patient_visit_vital, merge_hrp, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE'], how='left')

In [20]:
df_patient_visit_ehr

,PATIENT_STUDY_ID,ACCESSION_NUMBER,EXAM_COMPLETED_DATE,Study,Index,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,RISK_FACTOR_No family history of breast cancer,RISK_FACTOR_Personal breast cancer history,"RISK_FACTOR_Weak family history of breast cancer (aunt, grandmother, cousin)",RISK_FACTOR_Late child bearing (after 30),"RISK_FACTOR_Very strong family history of breast cancer (mother, sister, daughter, pre-menopause or multiple post-menopausal first degree relatives)","RISK_FACTOR_Family history of ovarian cancer in mother, sister, or daughter",RISK_FACTOR_Post-menopausal patient,RISK_FACTOR_BRCA1 gene mutation,RISK_FACTOR_Nulliparous,RISK_FACTOR_Family history of ovarian cancer in distant blood relatives,"RISK_FACTOR_Intermediate family history of breast cancer (mother, sister, daughter, post-menopause)","RISK_FACTOR_History of high risk lesion on previous biopsy, LCIS, atypical hyperplasia",RISK_FACTOR_Previous chest radiation therapy,RISK_FACTOR_Other family history of breast cancer,RISK_FACTOR_BRCA2 gene mutation,RISK_FACTOR_Family history unknown for breast cancer,RISK_FACTOR_History of ovarian cancer,RISK_FACTOR_History of endometrial cancer,RISK_FACTOR_Not available,BIRTH_DATE,GENDER_TITLE,RACE_KOREAN,RACE_WHITE,RACE_INDIAN (ASIAN),RACE_BLACK,RACE_UNREPORTED/CHOSE NOT TO DISCLOSE,RACE_OTHER ASIAN,RACE_VIETNAMESE,RACE_CHINESE,RACE_AMERICAN INDIAN/ALASKA NATIVE,RACE_FILIPINO,RACE_OTHER,RACE_JAPANESE,RACE_HAWAIIAN,RACE_OTHER PACIFIC ISLANDER,ETHNIC_NOT HISPANIC OR LATINO,"ETHNIC_ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN",ETHNIC_UNREPORTED/CHOSE NOT TO DISCLOSE,ETHNIC_HISPANIC OR LATINO,"ETHNIC_MEXICAN, MEXICAN AMERICAN, CHICANO",ETHNIC_PUERTO RICAN,PatientAge
0,4330344296,62861703,2019-12-20,SCREEN,INDEX-1,2019-09-06,2320.0,OZ,60.0,IN,28.32,9.0,19.0,0.0,40.0,0.0,40.0,3.0,3.0,NaN,POSTNAT,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1954-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,65.0
1,4330344296,67156697,2021-01-25,DIAG,INDEX,2021-02-23,2528.0,OZ,60.0,IN,30.86,9.0,19.0,0.0,40.0,0.0,40.0,3.0,3.0,NaN,POSTNAT,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1954-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,66.0
2,4330344296,67469599,2020-12-31,SCREEN,INDEX,2021-02-23,2528.0,OZ,60.0,IN,30.86,9.0,19.0,0.0,40.0,0.0,40.0,3.0,3.0,NaN,POSTNAT,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1954-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,66.0
3,4330358672,63576941,2019-06-14,DIAG,INDEX,2019-06-24,2352.0,OZ,68.0,IN,22.35,15.0,27.0,0.0,0.0,0.0,0.0,2.0,2.0,NaN,POSTNAT,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1968-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,50.0
4,4330358672,78682610,2018-05-21,SCREEN,INDEX-1,2018-07-09,2320.0,OZ,68.0,IN,22.05,15.0,27.0,0.0,0.0,0.0,0.0,2.0,2.0,NaN,POSTNAT,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1968-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,49.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1961,4339011646,63808536,2019-08-13,DIAG,INDEX,2019-08-13,2880.0,OZ,65.0,IN,29.95,13.0,39.0,52.0,0.0,0.0,0.0,2.0,3.0,NaN,POSTNAT,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1962-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,57.0
1962,4339011646,774

In [21]:
df_patient_visit_ehr.sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER'], ignore_index=True, inplace=True)

#### <span style="color:#FF6347;">**SAVE**</span> **mo_cancer_ehr, normal_ehr_@_index_1**

In [22]:
df_patient_visit_ehr["DATE_TAKEN"] = df_patient_visit_ehr["DATE_TAKEN"].dt.strftime("%Y-%m-%d")
df_patient_visit_ehr["EXAM_COMPLETED_DATE"] = df_patient_visit_ehr["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")

In [23]:
df_patient_visit_ehr['PATIENT_STUDY_ID'].nunique()

653

In [24]:
df_patient_ehr = df_patient_visit_ehr[df_patient_visit_ehr["Index"]=="INDEX-1"]

In [25]:
df_patient_ehr.shape, df_patient_ehr["PATIENT_STUDY_ID"].nunique()

((664, 63), 653)

In [26]:
df_patient_ehr = df_patient_ehr.drop_duplicates(
    subset=["PATIENT_STUDY_ID", "ACCESSION_NUMBER"], keep = 'first').reset_index(drop = True)

In [27]:
df_patient_ehr.shape, df_patient_ehr["PATIENT_STUDY_ID"].nunique()

((664, 63), 653)

In [28]:
dups = df_patient_ehr[df_patient_ehr.duplicated(subset='PATIENT_STUDY_ID', keep=False)].copy()

print(f"Rows involved    : {len(dups)}")
print(f"Unique dup IDs   : {dups['PATIENT_STUDY_ID'].nunique()}")
print()

# Show which columns differ between entries for the same patient
for pid, grp in dups.groupby('PATIENT_STUDY_ID'):
    varying = [c for c in grp.columns if grp[c].nunique(dropna=False) > 1]
    print(f"ID: {pid}  |  {len(grp)} rows  |  differing cols: {varying}")

Rows involved    : 22
Unique dup IDs   : 11

ID: 4333026733  |  2 rows  |  differing cols: ['ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE']
ID: 4333210683  |  2 rows  |  differing cols: ['ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE']
ID: 4333397770  |  2 rows  |  differing cols: ['ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE', 'DATE_TAKEN', 'HEIGHT']
ID: 4333505027  |  2 rows  |  differing cols: ['ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE', 'DATE_TAKEN', 'WEIGHT', 'BMI']
ID: 4333692906  |  2 rows  |  differing cols: ['ACCESSION_NUMBER']
ID: 4333878298  |  2 rows  |  differing cols: ['ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE']
ID: 4333972847  |  2 rows  |  differing cols: ['ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE']
ID: 4334111875  |  2 rows  |  differing cols: ['ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE']
ID: 4334460377  |  2 rows  |  differing cols: ['ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE', 'PatientAge']
ID: 4334469552  |  2 rows  |  differing cols: ['ACCESSION_NUMBER', 'EXAM_COMPLETED_DATE']
ID: 43346

In [29]:
dups

,PATIENT_STUDY_ID,ACCESSION_NUMBER,EXAM_COMPLETED_DATE,Study,Index,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,RISK_FACTOR_No family history of breast cancer,RISK_FACTOR_Personal breast cancer history,"RISK_FACTOR_Weak family history of breast cancer (aunt, grandmother, cousin)",RISK_FACTOR_Late child bearing (after 30),"RISK_FACTOR_Very strong family history of breast cancer (mother, sister, daughter, pre-menopause or multiple post-menopausal first degree relatives)","RISK_FACTOR_Family history of ovarian cancer in mother, sister, or daughter",RISK_FACTOR_Post-menopausal patient,RISK_FACTOR_BRCA1 gene mutation,RISK_FACTOR_Nulliparous,RISK_FACTOR_Family history of ovarian cancer in distant blood relatives,"RISK_FACTOR_Intermediate family history of breast cancer (mother, sister, daughter, post-menopause)","RISK_FACTOR_History of high risk lesion on previous biopsy, LCIS, atypical hyperplasia",RISK_FACTOR_Previous chest radiation therapy,RISK_FACTOR_Other family history of breast cancer,RISK_FACTOR_BRCA2 gene mutation,RISK_FACTOR_Family history unknown for breast cancer,RISK_FACTOR_History of ovarian cancer,RISK_FACTOR_History of endometrial cancer,RISK_FACTOR_Not available,BIRTH_DATE,GENDER_TITLE,RACE_KOREAN,RACE_WHITE,RACE_INDIAN (ASIAN),RACE_BLACK,RACE_UNREPORTED/CHOSE NOT TO DISCLOSE,RACE_OTHER ASIAN,RACE_VIETNAMESE,RACE_CHINESE,RACE_AMERICAN INDIAN/ALASKA NATIVE,RACE_FILIPINO,RACE_OTHER,RACE_JAPANESE,RACE_HAWAIIAN,RACE_OTHER PACIFIC ISLANDER,ETHNIC_NOT HISPANIC OR LATINO,"ETHNIC_ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN",ETHNIC_UNREPORTED/CHOSE NOT TO DISCLOSE,ETHNIC_HISPANIC OR LATINO,"ETHNIC_MEXICAN, MEXICAN AMERICAN, CHICANO",ETHNIC_PUERTO RICAN,PatientAge
19,4333026733,70264497,2017-11-13,SCREEN,INDEX-1,2017-08-07,1923.2,OZ,64.25,IN,20.47,13.0,0.0,51.0,0.0,0.0,0.0,0.0,0.0,NaN,POSTNAT,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1950-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,67.0
20,4333026733,79153553,2017-12-15,SCREEN,INDEX-1,2017-08-07,1923.2,OZ,64.25,IN,20.47,13.0,0.0,51.0,0.0,0.0,0.0,0.0,0.0,NaN,POSTNAT,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1950-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,67.0
99,4333210683,74504693,2016-07-30,SCREEN,INDEX-1,2016-07-22,2160.0,OZ,66.50,IN,21.46,11.0,0.0,51.0,0.0,0.0,0.0,0.0,0.0,NaN,POSTNAT,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1952-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,64.0
100,4333210683,74191819,2016-08-13,SCREEN,INDEX-1,2016-07-22,2160.0,OZ,66.50,IN,21.46,11.0,0.0,51.0,0.0,0.0,0.0,0.0,0.0,NaN,POSTNAT,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1952-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,64.0
168,4333397770,86620937,2016-05-09,SCREEN,INDEX-1,2016-05-18,1952.0,OZ,65.00,IN,20.30,12.0,23.0,59.0,0.0,0.0,0.0,3.0,5.0,NaN,POSTNAT,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1943-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,72.0
169,4333397770,75913994,2016-06-02,SCREEN,INDEX-1,2016-06-16,1952.0,OZ,NaN,IN,20.30,12.0,23.0,59.0,0.0,0.0,0.0,3.0,5.0,NaN,POSTNAT,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1943-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,72.0
188,4333505027,73880806,2017-01-25,SCREEN,INDEX-1,2016-05-06,1920.0,OZ,61.00,IN,22.69,13.0,0.0,49.0,0.0,0.0,0.0,0.0,0.0,NaN,POSTNAT,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1954-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0

In [30]:
ids_to_remove = [4333519498, 4334007595]
df_patient_visit_ehr = df_patient_visit_ehr[~df_patient_visit_ehr['PATIENT_STUDY_ID'].isin(ids_to_remove)]
df_patient_ehr = df_patient_ehr[~df_patient_ehr['PATIENT_STUDY_ID'].isin(ids_to_remove)]

In [31]:
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", 'mo_cancer_candidate_ehr' + ".xlsx")
df_patient_visit_ehr.to_excel(output_file, index=False)

In [32]:
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", 'mo_cancer_candidate_ehr_@_index_1' + ".xlsx")
# output_file = os.path.join("normal_candidate_ehr_@_index_1.xlsx")
df_patient_ehr.to_excel(output_file, index=False)

In [3]:
file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "MO cancer candidate", 'mo_cancer_candidate' + ".xlsx")
mo_cancer_candidate = pd.read_excel(file)
file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "MO cancer (default)", 'mo_cancer' + ".xlsx")
mo_cancer = pd.read_excel(file)

In [8]:
mo_cancer.shape, mo_cancer_candidate.shape

((1230, 19), (3436, 17))

In [5]:
# Filter df1 where both (pid, asc) pairs exist in df2
keys = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER']
overlapping_rows = mo_cancer_candidate.merge(mo_cancer[keys].drop_duplicates(), on=keys, how='inner')

# Save overlapping rows to a new Excel file
# overlapping_rows.to_excel('overlap_pid_asc_rows.xlsx', index=False)
# print(f"Found {len(overlapping_rows)} overlapping records.")

In [9]:
overlapping_rows.shape

(907, 17)

In [18]:
overlapping_pids = overlapping_rows["PATIENT_STUDY_ID"].unique()

In [19]:
mo_cancer_candidate_no_overlap = mo_cancer_candidate[~mo_cancer_candidate["PATIENT_STUDY_ID"].isin(overlapping_pids)]

In [ ]:
# mo_cancer_candidate_clean = mo_cancer_candidate.merge(
#     mo_cancer[keys].drop_duplicates(), 
#     on=keys, 
#     how='left', 
#     indicator=True
# )

# # 2. Keep only rows that exist exclusively in df2 ('left_only')
# mo_cancer_candidate_no_overlap = mo_cancer_candidate_clean[mo_cancer_candidate_clean['_merge'] == 'left_only'].drop(columns=['_merge'])

In [20]:
mo_cancer["PATIENT_STUDY_ID"].nunique(), mo_cancer_candidate["PATIENT_STUDY_ID"].nunique(), overlapping_rows["PATIENT_STUDY_ID"].nunique(), mo_cancer_candidate_no_overlap["PATIENT_STUDY_ID"].nunique()

(253, 653, 223, 430)

In [21]:
file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "MO cancer candidate", 'mo_cancer_candidate_no_overlap' + ".xlsx")
mo_cancer_candidate_no_overlap.to_excel(file, index=False)